Korzystanie z narzędzi generatywnej AI w rozwiązywaniu zadań nie jest dozwolone

<img src="no_AI.png" alt="Use of AI allowed only when properly documented " width="100" height="100">

# Zadanie obowiązkowe [0-10] pkt

1. [0-1.5 pkt] Porównaj wyniki k-means z k-medoids, testując przynajmniej trzy różne metryki (użyj [tego API](https://scikit-learn-extra.readthedocs.io/en/stable/generated/sklearn_extra.cluster.KMedoids.html)). Jako miar porównania metod, użyj [ARI](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html) oraz [Silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html) (dotyczy to też punktów niżej). Wyrysuj klastry i omów wyniki.
1. [0-2 pkt] Użyj DSBSCAN i wyrysuj klastry dla różnych kombinacji wartości `eps`, `min_samples` i `metric`. Porównaj wyniki z metodami wyżej.
2. [0-2 pkt] Użyj HDBSCAN, testując różne kombinacje parametrów. Czy wyniki różnią się od DBSCAN?
3. [0-2.5 pkt] Uzyj [klastrowania aglomeracyjnego](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html). Dla każdej wartości parametru `linkage`, wyrysuj [dendrogram](https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html). Na podstawie dendrogramów, dobierz, w Twojej ocenie, optymalne parametry `n_clusters` albo `distance_threshold` (chodzi o określenie, czy w dendrogramie, na pewnym poziomie odcięcia, widać ewidentne klastry). Wyrysuj klastry i skomentuj wyniki.  
1. [0-1 pkt] Sprawdź wyniki poszczególnych metod klastrowania bez skalowania cech. Zinterpretuj wyniki.
1. [0-1 pkt] Który z algorytmów działa najlepiej dla tego problemu? Skomentuj wyniki.

# Zadanie dodatkowe [0-10] pkt

1. [0-7 pkt] Wytrenuj sieć konwolucyjną (CNN) na macierzach podobieństwa, przewidując klasy CATH (problem klasyfikacji). Pamiętaj o ewaluacji na zbiorze walidacyjnym.
2. [0-1 pkt] Uruchom wytrenowany model na wszystkich białkach. Użyj wag z ostatniej warstwy jako wektora cech.
3. [0-1 pkt] Dokonaj redukcji wymiarowości wektora cech do dwóch wymiarów np. za pomocą PCA.
4. [0-1 pkt] Czy tak skonstruowane cechy korelują z klasą białek?

#### przygotowanie struktury projektu

In [6]:
import zipfile
import os

zip_filename = "klasteryzacja_data.zip"

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(".")

print("projekt rozpakowany")
print(os.listdir("."))

projekt rozpakowany
['.config', 'klasteryzacja_data.zip', 'klasteryzacja_data', 'sample_data']


In [7]:
from pathlib import Path
main_path = Path("klasteryzacja_data")
coord_dir = main_path / "coords"
metadata_path = main_path /"cath" / "metadata.txt"
features_path = main_path / "features.npz"

print(coord_dir.exists())
print(metadata_path.exists())
print(features_path.exists())

True
True
True


#### importy

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import networkx as nx
import seaborn as sns

from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.cluster import KMeans

#### features, metadata

In [9]:
feature_vectors = np.load(features_path)
feature_vectors = {key: feature_vectors[key] for key in feature_vectors.files}
print(len(feature_vectors))

columns = ["num_nodes","num_edges","mean_degree","var_degree","mean_weighted_degree","density","avg_neighbor_degree"]

data = pd.DataFrame.from_dict(feature_vectors, orient="index", columns=columns)
data.head()

6631


,num_nodes,num_edges,mean_degree,var_degree,mean_weighted_degree,density,avg_neighbor_degree
16pkA01,1424.0,150626.0,211.553371,5371.828612,145.931023,0.148667,227.330618
16vpA00,2457.0,270770.0,220.407000,4539.048840,151.953644,0.089742,232.876013
1a0rP02,451.0,30414.0,134.873614,2005.955202,96.484231,0.299719,145.013901
1a1xA00,881.0,84571.0,191.988649,5082.678645,133.416658,0.218169,209.078473
1a3qA01,1398.0,135551.0,193.921316,4513.348601,134.531393,0.138813,207.707550


In [10]:
metadata = pd.read_csv(metadata_path, delimiter = '\t', header = None, names = ['protein','C','A','T','H'], index_col = 'protein')
print(len(metadata))
metadata.head()

6631


,C,A,T,H
protein,,,,
1oaiA00,1,10,8,10
1oizB01,1,10,8,20
2j5yA00,1,10,8,40
6bekD00,1,10,8,50
5ep2A02,1,10,8,60


In [13]:
data.join(metadata)
print(data.shape)
data.head()

(6631, 7)


,num_nodes,num_edges,mean_degree,var_degree,mean_weighted_degree,density,avg_neighbor_degree
16pkA01,1424.0,150626.0,211.553371,5371.828612,145.931023,0.148667,227.330618
16vpA00,2457.0,270770.0,220.407000,4539.048840,151.953644,0.089742,232.876013
1a0rP02,451.0,30414.0,134.873614,2005.955202,96.484231,0.299719,145.013901
1a1xA00,881.0,84571.0,191.988649,5082.678645,133.416658,0.218169,209.078473
1a3qA01,1398.0,135551.0,193.921316,4513.348601,134.531393,0.138813,207.707550


#### przygotowanie bazy do klastrowania

In [14]:
from sklearn.metrics import silhouette_score, adjusted_rand_score

feature_cols = ["num_nodes","num_edges","mean_degree","var_degree","mean_weighted_degree","density","avg_neighbor_degree"]

X_raw = data[feature_cols].copy()
y_true = data['C'].astype(str)

print(f"Protein number: {len(X_raw)}")
print(f"Number of unique classes CATH (C): {y_true.nunique()}")
print((y_true.value_counts()))

KeyError: 'C'